# Experiment 9: Random Forest Algorithm for Classification
### Dataset: Pima Indians Diabetes Dataset
**Input (X):** 8 Medical Features  
**Output (Y):** Outcome → 0 = No Diabetes, 1 = Diabetes

## Step 1 & 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully.')

## Step 3: Load Diabetes Dataset

In [ ]:
# Place diabetes.csv in the same folder as this notebook
df = pd.read_csv('diabetes.csv')

print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## Step 4: Clean and Preprocess Data

In [ ]:
print('Dataset Info:')
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

In [ ]:
# Columns where 0 is biologically invalid — replace with column median
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in zero_invalid_cols:
    zero_count = (df[col] == 0).sum()
    median_val = df[df[col] != 0][col].median()
    df[col] = df[col].replace(0, median_val)
    print(f'{col}: replaced {zero_count} zero(s) with median ({median_val})')

print('\nData cleaned successfully.')

In [ ]:
# Class distribution
print('Outcome Distribution:')
print(df['Outcome'].value_counts())

plt.figure(figsize=(5, 4))
sns.countplot(x='Outcome', data=df, palette=['steelblue', 'salmon'])
plt.title('Class Distribution')
plt.xticks([0, 1], ['No Diabetes (0)', 'Diabetes (1)'])
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Step 5: Separate X (Features) and Y (Target)

In [ ]:
X = df.drop('Outcome', axis=1)   # 8 input features
y = df['Outcome']                 # Target: 0 or 1

print(f'X shape (features): {X.shape}')
print(f'y shape (target):   {y.shape}')
print(f'\nFeatures: {list(X.columns)}')

## Step 6: Split into Training and Testing Sets (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Testing set size  : {X_test.shape[0]} samples')

## Step 7: Initialize Random Forest Classifier

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=100,   # Number of trees
    max_depth=None,     # Grow trees fully
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

print(f'Random Forest Classifier initialized.')
print(f'Number of trees: {rf_classifier.n_estimators}')

## Step 8: Train Multiple Trees

In [ ]:
rf_classifier.fit(X_train, y_train)

print(f'Model trained successfully.')
print(f'Number of trees trained: {len(rf_classifier.estimators_)}')

## Step 9 & 10: Apply Majority Voting — Predict Outcome

In [ ]:
# Random Forest internally applies majority voting across all 100 trees
y_pred = rf_classifier.predict(X_test)

print('Sample Result Table (first 10 predictions):')
print('-' * 25)
print(f'{"Actual":<12} {"Predicted":<12}')
print('-' * 25)
for actual, predicted in zip(y_test.values[:10], y_pred[:10]):
    print(f'{actual:<12} {predicted:<12}')
print('-' * 25)

## Step 11: Evaluate Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Diabetes', 'Diabetes'])

plt.figure(figsize=(6, 5))
disp.plot(cmap='Blues', colorbar=False)
plt.title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (TN): {tn}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Positives  (TP): {tp}')

In [ ]:
# Feature Importance
feat_imp_df = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': rf_classifier.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nFeature Importance Rankings:')
print(feat_imp_df.to_string(index=False))

## Output Interpretation

| Value | Meaning      |
|-------|--------------|
| 0     | No Diabetes  |
| 1     | Diabetes     |

In [ ]:
print(f'Final Model Accuracy: {accuracy * 100:.2f}%')
print('\nStep 12: Stop')